In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio as rio
from matplotlib import pyplot as plt
import os
from tqdm.autonotebook import tqdm

C:\Users\Administrator\AppData\Local\Temp\ipykernel_31424\3405216990.py:7: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
country_raster_1km_ = rio.open(r'..\results\country-rasters\country-raster-eck4-1km.tif')
country_raster_10km_ = rio.open(r'..\results\country-rasters\country-raster-eck4-10km.tif')
country_raster_50km_ = rio.open(r'..\results\country-rasters\country-raster-eck4-50km.tif')

In [5]:
def upscale_AOH(taxon, new_res):
    scale_dict = {'10km': 10, '50km': 50}
    scale = scale_dict[new_res]
    
    country_raster_1_ = rio.open('../results/country-rasters/country-raster-eck4-1km.tif')
    country_raster_2_ = rio.open(f'../results/country-rasters/country-raster-eck4-{new_res}.tif')

    a = int(abs(country_raster_1_.transform[5] - country_raster_2_.transform[5]) / 1e3)   # the number of offset grid cells along the row direction, under the original resolution of 1km 
    b = int(abs(country_raster_1_.transform[2] - country_raster_2_.transform[2]) / 1e3)   # the number of offset grid cells along the col direction, under the original resolution of 1km

    catalogue = pd.read_csv(f'../results/area-of-habitat-1km/catalogue-{taxon}.csv')
    valid_species = catalogue[catalogue['have_a_AOH'] == True]
    new_catalogue = valid_species.copy()
    new_catalogue['row_min'] = np.nan
    new_catalogue['row_max'] = np.nan
    new_catalogue['col_min'] = np.nan
    new_catalogue['col_max'] = np.nan

    for i in tqdm(valid_species.index):
        row_min_src = catalogue.loc[i, 'row_min']
        row_max_src = catalogue.loc[i, 'row_max']
        col_min_src = catalogue.loc[i, 'col_min']
        col_max_src = catalogue.loc[i, 'col_max']

        row_min_dst = (row_min_src + a) // scale
        row_max_dst = (row_max_src + a) // scale
        col_min_dst = (col_min_src + b) // scale
        col_max_dst = (col_max_src + b) // scale

        # print(row_min_src, row_max_src, col_min_src, col_max_src)
        # print(row_min_dst, row_max_dst, col_min_dst, col_max_dst)

        # print(949*10, 9481 + 9, 950*10, 963*10, 9623+9, 964*10)
        # print(2756*10, 27558 + 8, 2757*10, 2771*10, 27705+8, 2772*10)

        offset_x = col_min_src + b - col_min_dst * scale
        offset_y = row_min_src + a - row_min_dst * scale
        # print(offset_x, offset_y)

        a_species = catalogue.loc[i, 'name']
        a_src = rio.open(f'../results/area-of-habitat-1km/{taxon}/{a_species}.tif')
        a_AOH_src = a_src.read(1)

        H_src = row_max_src - row_min_src + 1
        W_src = col_max_src - col_min_src + 1
        H_dst = row_max_dst - row_min_dst + 1
        W_dst = col_max_dst - col_min_dst + 1
        # a_AOH_dst = np.zeros(shape=(H_dst, W_dst))
        a_AOH_mediate = np.zeros(shape=(H_dst * scale, W_dst * scale))
        # print(a_AOH_dst_to_1km.shape)
        a_AOH_mediate[offset_y:offset_y + H_src, offset_x:offset_x + W_src] = a_AOH_src
        # a_AOH_dst = a_AOH_mediate.reshape(H_dst, scale, W_dst, scale).max(axis=(1, 3)).astype(np.uint8)
        a_AOH_dst = a_AOH_mediate.reshape(H_dst, scale, W_dst, scale).sum(axis=(1, 3)).astype(np.uint16)

        transform_10km = country_raster_2_.transform

        x0 = transform_10km.c + col_min_dst * transform_10km.a
        y0 = transform_10km.f + row_min_dst * transform_10km.e

        new_transform = rio.Affine(transform_10km.a, 0, x0,
                                0, transform_10km.e, y0)

        output_path = os.path.join('..', 'results', f'area-of-habitat-{new_res}', taxon, f'{a_species}.tif')
        with rio.open(
                output_path,
                'w',
                driver='GTiff',
                height=a_AOH_dst.shape[0],
                width=a_AOH_dst.shape[1],
                count=1,
                dtype=a_AOH_dst.dtype, 
                crs=country_raster_2_.crs,
                transform=new_transform,
                compress='lzw',
            ) as dst:
                dst.write(a_AOH_dst, 1)

        new_catalogue.loc[i, 'row_min'] = int(row_min_dst)
        new_catalogue.loc[i, 'row_max'] = int(row_max_dst)
        new_catalogue.loc[i, 'col_min'] = int(col_min_dst)
        new_catalogue.loc[i, 'col_max'] = int(col_max_dst)
        new_catalogue.to_csv(os.path.join('..', 'results', f'area-of-habitat-{new_res}', f'catalogue-{taxon}.csv'), index=False)

In [6]:
upscale_AOH('mammals', '50km')

  0%|          | 0/5237 [00:00<?, ?it/s]

In [8]:
gg = rio.open('../results/area-of-habitat-50km/mammals/Abditomys latidens.tif').read(1)

In [12]:
upscale_AOH('birds', '50km')

  0%|          | 0/9927 [00:00<?, ?it/s]

In [13]:
upscale_AOH('reptiles', '50km')

  0%|          | 0/7948 [00:00<?, ?it/s]

In [14]:
upscale_AOH('amphibians', '50km')

  0%|          | 0/6215 [00:00<?, ?it/s]